# Bölüm 6: Kategorik Veriler için Çıkarım
**Haydar Kılıç — Yapay Zeka Mühendisliği**

Bu notebook, aşağıdaki konuları uygulamalı olarak ele almaktadır:

1. **Tek Oran için Çıkarım** — Güven aralığı ve hipotez testi
2. **İki Oranın Farkı** — Karşılaştırmalı çıkarım
3. **Uyum İyiliği Ki-Kare Testi** — Gözlemlenen vs beklenen dağılım
4. **Bağımsızlık Ki-Kare Testi** — İki kategorik değişken arasındaki ilişki

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import norm, chi2
import warnings
warnings.filterwarnings('ignore')

# Grafik ayarları
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("Kütüphaneler başarıyla yüklendi ✓")

---
## 1. Tek Oran için Çıkarım

### 1.1 Teorik Temel

Bir kitle oranı $p$ için örneklem oranı $\hat{p}$, **Merkezi Limit Teoremi** gereği yaklaşık normal dağılır:

$$\hat{p} \sim N\left(p,\ SH = \sqrt{\frac{p(1-p)}{n}}\right)$$

**Koşullar:**
- Bağımsızlık: Rassal örneklem + %10 koşulu
- Başarı-başarısızlık: En az 10 başarı ve 10 başarısızlık

**%95 Güven Aralığı:**
$$\hat{p} \pm 1.96 \times \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

### 1.2 GSS Örneği — Deneysel Tasarım Sorusu

GSS anketi, 670 Amerikalının 571'inin (%85) deneysel tasarım sorusunu doğru yanıtladığını buldu. Tüm Amerikalıların bu soruyu doğru yanıtlama oranı için %95 güven aralığı hesaplayalım.

In [ ]:
# ── Veriler ──────────────────────────────────────────────────────────────────
n = 670          # örneklem büyüklüğü
basari = 571     # doğru yanıtlayan kişi sayısı
p_hat = basari / n
guven_duzeyi = 0.95
z_star = norm.ppf(1 - (1 - guven_duzeyi) / 2)   # kritik değer

# ── Koşul Kontrolü ───────────────────────────────────────────────────────────
print("=" * 50)
print("KOŞUL KONTROLÜ")
print("=" * 50)
print(f"Başarı sayısı  : {basari}  {'✓ (≥10)' if basari >= 10 else '✗ (<10)'}")
print(f"Başarısızlık   : {n - basari}  {'✓ (≥10)' if (n - basari) >= 10 else '✗ (<10)'}")
print(f"Örneklem oranı : {p_hat:.4f}")

# ── Standart Hata ────────────────────────────────────────────────────────────
SH = np.sqrt(p_hat * (1 - p_hat) / n)
hata_payi = z_star * SH
GA_alt = p_hat - hata_payi
GA_ust = p_hat + hata_payi

print("\n" + "=" * 50)
print("HESAPLAMALAR")
print("=" * 50)
print(f"Kritik değer (z*)  : {z_star:.4f}")
print(f"Standart hata (SH) : {SH:.4f}")
print(f"Hata payı (HP)     : ±{hata_payi:.4f}")
print(f"\n%95 Güven Aralığı  : ({GA_alt:.4f}, {GA_ust:.4f})")
print(f"                   = (%{GA_alt*100:.1f}, %{GA_ust*100:.1f})")

In [ ]:
# ── Görselleştirme: Güven Aralığı ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

x = np.linspace(0.78, 0.92, 400)
y = norm.pdf(x, p_hat, SH)

# Dağılım eğrisi
ax.plot(x, y, 'steelblue', lw=2.5)

# Güven aralığı gölgelendirmesi
x_fill = np.linspace(GA_alt, GA_ust, 300)
ax.fill_between(x_fill, norm.pdf(x_fill, p_hat, SH),
                alpha=0.35, color='steelblue', label='%95 GA')

# Sınır çizgileri
for val, label in [(GA_alt, f'{GA_alt:.3f}'), (GA_ust, f'{GA_ust:.3f}')]:
    ax.axvline(val, color='navy', lw=1.5, ls='--')
    ax.text(val, ax.get_ylim()[1] * 0.02, label,
            ha='center', va='bottom', fontsize=10, color='navy')

ax.axvline(p_hat, color='crimson', lw=2, label=f'$\\hat{{p}}$ = {p_hat:.3f}')

ax.set_xlabel('Örneklem Oranı ($\\hat{p}$)', fontsize=12)
ax.set_ylabel('Yoğunluk', fontsize=12)
ax.set_title('%95 Güven Aralığı — Deneysel Tasarım Sorusu (n=670)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nYorum: Tüm Amerikalıların %{GA_alt*100:.1f} ile %{GA_ust*100:.1f}'i arasındaki bir oranının")
print("deneysel tasarım sorusunu doğru yanıtladığına %95 güvenle inanıyoruz.")

### 1.3 Hipotez Testi — Tek Oran

**Soru:** Bu veriler, Amerikalıların %80'inden fazlasının deneysel tasarım hakkında iyi sezgiye sahip olduğuna dair ikna edici kanıt sunuyor mu?

$$H_0: p = 0.80 \qquad H_A: p > 0.80$$

**Not:** Hipotez testinde standart hata, $H_0$'daki $p_0$ değeri kullanılarak hesaplanır.

In [ ]:
# ── Hipotez Testi ─────────────────────────────────────────────────────────────
p_0 = 0.80    # H0'daki sıfır değeri
p_hat = 571 / 670
n = 670

# Koşul kontrolü (H0 değeri kullanılır)
beklenen_basari = n * p_0
beklenen_basarisizlik = n * (1 - p_0)
print("KOŞUL KONTROLÜ (HT için H0 değeriyle):")
print(f"  Beklenen başarı    : {beklenen_basari:.0f}  {'✓' if beklenen_basari >= 10 else '✗'}")
print(f"  Beklenen başarısızlık: {beklenen_basarisizlik:.0f}  {'✓' if beklenen_basarisizlik >= 10 else '✗'}")

# Test istatistiği
SH_ht = np.sqrt(p_0 * (1 - p_0) / n)
Z = (p_hat - p_0) / SH_ht

# p-değeri (sağ kuyruk testi)
p_deger = 1 - norm.cdf(Z)

print("\nHİPOTEZ TESTİ")
print(f"  H0 : p = {p_0}")
print(f"  HA : p > {p_0}")
print(f"  SH = sqrt({p_0}×{1-p_0}/{n}) = {SH_ht:.4f}")
print(f"  Z  = ({p_hat:.4f} - {p_0}) / {SH_ht:.4f} = {Z:.2f}")
print(f"  p-değeri = P(Z > {Z:.2f}) = {p_deger:.4f}")
print(f"\n  Karar (α=0.05): {'H0 REDDEDİLİR ✓' if p_deger < 0.05 else 'H0 reddedilemez'}")

In [ ]:
# ── Görselleştirme: Z-testi ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

x = np.linspace(-4, 5, 400)
y = norm.pdf(x)

ax.plot(x, y, 'steelblue', lw=2.5)

# p-değeri alanı (sağ kuyruk)
x_fill = np.linspace(Z, 5, 200)
ax.fill_between(x_fill, norm.pdf(x_fill), alpha=0.45,
                color='crimson', label=f'p-değeri = {p_deger:.4f}')

ax.axvline(Z, color='crimson', lw=2, ls='--', label=f'Z = {Z:.2f}')
ax.axvline(0, color='gray', lw=1, ls=':')

ax.set_xlabel('Z değeri', fontsize=12)
ax.set_ylabel('Yoğunluk', fontsize=12)
ax.set_title('Hipotez Testi: $H_0: p=0.80$ vs $H_A: p>0.80$', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Yorum: p-değeri < 0.05 olduğundan H0'ı reddediyoruz.")
print("Veriler, Amerikalıların %80'inden fazlasının deneysel tasarım")
print("hakkında iyi sezgiye sahip olduğuna dair ikna edici kanıt sunmaktadır.")

### 1.4 Örneklem Büyüklüğü Seçimi

Hata payını belirli bir değere indirmek için gereken örneklem büyüklüğü:

$$n \geq \frac{z^{*2} \cdot \hat{p}(1-\hat{p})}{HP^2}$$

**Önceki çalışma yoksa:** $\hat{p} = 0.5$ kullanılır (en tutucu tahmin, maksimum $n$).

In [ ]:
# ── Örneklem Büyüklüğü Hesaplama ─────────────────────────────────────────────
import math

def orneklem_buyuklugu(HP, guven=0.95, p_hat=None):
    """
    HP      : İstenen hata payı (örn. 0.01 = ±%1)
    guven   : Güven düzeyi
    p_hat   : Önceki tahmin; None ise p_hat=0.5 (tutucu)
    """
    z = norm.ppf(1 - (1 - guven) / 2)
    if p_hat is None:
        p_hat = 0.5
        not_str = "(p̂=0.5 kullanıldı — tutucu tahmin)"
    else:
        not_str = f"(p̂={p_hat} kullanıldı — önceki çalışmadan)"
    n = (z**2 * p_hat * (1 - p_hat)) / HP**2
    return math.ceil(n), not_str

# Örnek 1: HP = %1, önceki p̂ = 0.85
n1, not1 = orneklem_buyuklugu(HP=0.01, p_hat=0.85)
# Örnek 2: HP = %1, önceki çalışma yok
n2, not2 = orneklem_buyuklugu(HP=0.01, p_hat=None)
# Örnek 3: HP = %3
n3, not3 = orneklem_buyuklugu(HP=0.03, p_hat=None)

print("ÖRNEKLEM BÜYÜKLÜĞÜ HESAPLAMA (%95 güven düzeyi)")
print("-" * 60)
print(f"HP = ±%1, {not1}")
print(f"  → En az n = {n1} kişi örneklenmeli\n")
print(f"HP = ±%1, {not2}")
print(f"  → En az n = {n2} kişi örneklenmeli\n")
print(f"HP = ±%3, {not3}")
print(f"  → En az n = {n3} kişi örneklenmeli")

In [ ]:
# ── p̂ değerine göre gerekli n — görselleştirme ───────────────────────────────
p_values = np.linspace(0.01, 0.99, 200)
z = norm.ppf(0.975)   # %95 için z*=1.96

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, HP in enumerate([0.01, 0.03]):
    n_vals = np.ceil((z**2 * p_values * (1 - p_values)) / HP**2)
    axes[i].plot(p_values, n_vals, 'steelblue', lw=2.5)
    axes[i].axvline(0.5, color='crimson', lw=1.5, ls='--', label='p̂=0.5 (maks)')
    axes[i].fill_between(p_values, n_vals, alpha=0.15, color='steelblue')
    axes[i].set_xlabel('$\\hat{p}$ (Tahmin edilen oran)', fontsize=11)
    axes[i].set_ylabel('Gerekli n', fontsize=11)
    axes[i].set_title(f'HP = ±%{int(HP*100)} için Gerekli Örneklem Büyüklüğü', fontsize=12)
    axes[i].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Not: p̂=0.5, en yüksek örneklem büyüklüğünü verir → en tutucu tahmin.")

---
## 2. İki Oranın Farkı

### 2.1 Teorik Temel

İki bağımsız grup için oranların farkının standart hatası:

$$SH_{(\hat{p}_1 - \hat{p}_2)} = \sqrt{\frac{p_1(1-p_1)}{n_1} + \frac{p_2(1-p_2)}{n_2}}$$

**Güven aralığı:** $(\hat{p}_1 - \hat{p}_2) \pm z^* \times SH$

**Hipotez testinde** ($H_0: p_1 = p_2$) **havuzlanmış oran** kullanılır:

$$\hat{p}_{havuz} = \frac{\text{başarı}_1 + \text{başarı}_2}{n_1 + n_2}$$

### 2.2 Eriyen Buz Tabakası — Duke vs ABD

In [ ]:
# ── Veriler ──────────────────────────────────────────────────────────────────
duke_cok = 69;   duke_toplam = 105
abd_cok  = 454;  abd_toplam  = 680

p_duke = duke_cok / duke_toplam
p_abd  = abd_cok  / abd_toplam
fark   = p_duke - p_abd

# Tablo özeti
df_veri = pd.DataFrame({
    'Grup': ['Duke Öğrencileri', 'ABD Geneli'],
    'Çok Rahatsız': [duke_cok, abd_cok],
    'Toplam (n)': [duke_toplam, abd_toplam],
    'Oran (p̂)': [f'{p_duke:.3f}', f'{p_abd:.3f}']
})
print(df_veri.to_string(index=False))
print(f"\nOran farkı: p̂_Duke - p̂_ABD = {p_duke:.3f} - {p_abd:.3f} = {fark:.3f}")

In [ ]:
# ── %95 Güven Aralığı ─────────────────────────────────────────────────────────
z_star = norm.ppf(0.975)

SH_fark = np.sqrt(
    p_duke * (1 - p_duke) / duke_toplam +
    p_abd  * (1 - p_abd)  / abd_toplam
)

GA_alt = fark - z_star * SH_fark
GA_ust = fark + z_star * SH_fark

print("İKİ ORAN FARKI — GÜVEN ARALIĞI")
print(f"  SH = sqrt({p_duke:.3f}×{1-p_duke:.3f}/{duke_toplam} + {p_abd:.3f}×{1-p_abd:.3f}/{abd_toplam})")
print(f"     = {SH_fark:.4f}")
print(f"  HP = {z_star:.2f} × {SH_fark:.4f} = ±{z_star*SH_fark:.4f}")
print(f"\n  %95 GA = ({GA_alt:.3f}, {GA_ust:.3f})")
print(f"\n  Yorum: Güven aralığı 0'ı kapsıyor mu? {'EVET' if GA_alt < 0 < GA_ust else 'HAYIR'}")
if GA_alt < 0 < GA_ust:
    print("  → İki oran arasında istatistiksel olarak anlamlı fark YOK.")

In [ ]:
# ── Hipotez Testi — Havuzlanmış Oran ─────────────────────────────────────────
p_havuz = (duke_cok + abd_cok) / (duke_toplam + abd_toplam)

SH_havuz = np.sqrt(
    p_havuz * (1 - p_havuz) / duke_toplam +
    p_havuz * (1 - p_havuz) / abd_toplam
)

Z_ht = fark / SH_havuz
p_deger = 2 * norm.cdf(abs(Z_ht), loc=0)   # iki kuyruklu
p_deger = 2 * (1 - norm.cdf(abs(Z_ht)))    # sağ + sol kuyruk

print("HİPOTEZ TESTİ — HAVUZLANMIŞ ORAN")
print(f"  H0 : p_Duke = p_ABD")
print(f"  HA : p_Duke ≠ p_ABD")
print(f"\n  Havuzlanmış oran:")
print(f"  p̂_havuz = ({duke_cok}+{abd_cok}) / ({duke_toplam}+{abd_toplam}) = {p_havuz:.3f}")
print(f"  SH      = {SH_havuz:.4f}")
print(f"  Z       = {fark:.3f} / {SH_havuz:.4f} = {Z_ht:.2f}")
print(f"  p-değeri = 2×P(Z < {Z_ht:.2f}) = {p_deger:.4f}")
print(f"\n  Karar (α=0.05): {'H0 REDDEDİLİR' if p_deger < 0.05 else 'H0 reddedilemez ✓'}")
print("  Yorum: İki grup arasında anlamlı bir oran farkı bulunamamıştır.")

In [ ]:
# ── GA vs HT Standart Hata Karşılaştırması ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

x = np.linspace(-0.15, 0.15, 400)

# GA için dağılım
y_ga = norm.pdf(x, fark, SH_fark)
ax.plot(x, y_ga, 'steelblue', lw=2.5, label=f'GA dağılımı (SH={SH_fark:.4f})')

# GA gölgeleme
x_fill = np.linspace(GA_alt, GA_ust, 300)
ax.fill_between(x_fill, norm.pdf(x_fill, fark, SH_fark),
                alpha=0.25, color='steelblue', label='%95 GA')

# Sıfır çizgisi
ax.axvline(0, color='crimson', lw=2, ls='--', label='0 (fark yok)')
ax.axvline(fark, color='navy', lw=2, label=f'Gözlemlenen fark = {fark:.3f}')

# GA sınırları
for val in [GA_alt, GA_ust]:
    ax.axvline(val, color='steelblue', lw=1, ls=':')
    ax.text(val, -0.3, f'{val:.3f}', ha='center', fontsize=9, color='steelblue')

ax.set_xlabel('$\\hat{p}_{Duke} - \\hat{p}_{ABD}$', fontsize=12)
ax.set_title('Duke vs ABD — İki Oran Farkı için %95 Güven Aralığı', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(bottom=-0.5)
plt.tight_layout()
plt.show()

### 2.3 Referans: GA vs HT Standart Hatası Farkı

| | Güven Aralığı (GA) | Hipotez Testi (HT) |
|---|---|---|
| **Tek oran** | $SH = \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$ | $SH = \sqrt{\frac{p_0(1-p_0)}{n}}$ |
| **İki oran** | $SH = \sqrt{\frac{\hat{p}_1(1-\hat{p}_1)}{n_1} + \frac{\hat{p}_2(1-\hat{p}_2)}{n_2}}$ | $SH = \sqrt{\hat{p}_{havuz}(1-\hat{p}_{havuz})\left(\frac{1}{n_1}+\frac{1}{n_2}\right)}$ |

---
## 3. Uyum İyiliği Ki-Kare Testi

### 3.1 Teorik Temel

Gözlemlenen sayıların beklenen dağılıma uyumunu ölçmek için **ki-kare (χ²)** istatistiği kullanılır:

$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}$$

- $O_i$: Gözlemlenen sayı
- $E_i$: Beklenen sayı  
- $k$: Kategori sayısı
- **Serbestlik derecesi:** $sd = k - 1$
- **p-değeri:** χ² eğrisinin sağ kuyruğu

### 3.2 Ki-Kare Dağılımı — Görselleştirme

In [ ]:
# ── Ki-Kare Dağılımı — Farklı sd Değerleri ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

x = np.linspace(0.01, 30, 500)
renkler = ['#1f77b4', '#2ca02c', '#d62728']
stiller = ['-', '--', ':']

for sd, renk, stil in zip([2, 4, 9], renkler, stiller):
    y = chi2.pdf(x, df=sd)
    ax.plot(x, y, color=renk, lw=2.5, ls=stil, label=f'sd = {sd}')

ax.set_xlim(0, 28)
ax.set_ylim(0, 0.45)
ax.set_xlabel('χ²', fontsize=13)
ax.set_ylabel('Yoğunluk', fontsize=13)
ax.set_title('Ki-Kare Dağılımı — Farklı Serbestlik Dereceleri', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, title='Serbestlik Derecesi')

# Açıklama notları
ax.annotate('sd arttıkça:\n• Merkez sağa kayar\n• Değişkenlik artar\n• Normale yaklaşır',
            xy=(18, 0.25), fontsize=10,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

# sd → ∞ ile normale yakınsama
print("NOT: sd büyüdükçe χ² dağılımı normale yaklaşır.")
print("     χ²'nin beklenen değeri = sd, varyansı = 2×sd")

### 3.3 Labby'nin Zarları

In [ ]:
# ── Labby Zar Deneyi Verileri ─────────────────────────────────────────────────
sonuclar = [1, 2, 3, 4, 5, 6]
gozlemlenen = np.array([53222, 52118, 52465, 52338, 52244, 53285])
toplam_atis = gozlemlenen.sum()

# Eşit olasılık varsayımı altında beklenen
beklenen = np.array([toplam_atis / 6] * 6)

# Ki-kare katkıları
katkılar = (gozlemlenen - beklenen)**2 / beklenen
chi2_stat = katkılar.sum()
sd = len(sonuclar) - 1
p_deger_zar = chi2.sf(chi2_stat, df=sd)   # sf = 1 - cdf = sağ kuyruk

# Özet tablo
df_zar = pd.DataFrame({
    'Sonuç': sonuclar,
    'Gözlemlenen (O)': gozlemlenen,
    'Beklenen (E)': beklenen.astype(int),
    'O - E': (gozlemlenen - beklenen).astype(int),
    '(O-E)²/E': katkılar.round(2)
})

print("LABBY'NİN ZARLARI — Uyum İyiliği Testi")
print(df_zar.to_string(index=False))
print("-" * 55)
print(f"χ² = {chi2_stat:.2f}   sd = {sd}   p-değeri = {p_deger_zar:.6f}")
print(f"\nKarar (α=0.05): {'H0 REDDEDİLİR — Zarlar yanlıdır!' if p_deger_zar < 0.05 else 'H0 reddedilemez — Zarlar adil görünüyor'}")

In [ ]:
# ── Görselleştirme: Ki-Kare p-değeri ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Sol: Ki-kare dağılımı ve p-değeri
ax = axes[0]
x = np.linspace(0, 35, 500)
y = chi2.pdf(x, df=sd)
ax.plot(x, y, 'steelblue', lw=2.5)

x_fill = np.linspace(chi2_stat, 35, 300)
ax.fill_between(x_fill, chi2.pdf(x_fill, df=sd),
                alpha=0.5, color='crimson', label=f'p-değeri = {p_deger_zar:.5f}')
ax.axvline(chi2_stat, color='crimson', lw=2, ls='--', label=f'χ² = {chi2_stat:.2f}')
ax.set_xlabel('χ²', fontsize=12)
ax.set_ylabel('Yoğunluk', fontsize=12)
ax.set_title(f'Ki-Kare Dağılımı (sd={sd})', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Sağ: Gözlemlenen vs Beklenen çubuk grafik
ax2 = axes[1]
x_pos = np.arange(len(sonuclar))
genislik = 0.35
bars1 = ax2.bar(x_pos - genislik/2, gozlemlenen, genislik,
                color='steelblue', label='Gözlemlenen', alpha=0.85)
bars2 = ax2.bar(x_pos + genislik/2, beklenen, genislik,
                color='gray', label='Beklenen', alpha=0.85)
ax2.axhline(beklenen[0], color='gray', lw=1.5, ls='--', alpha=0.7)
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'Yüz {s}' for s in sonuclar])
ax2.set_ylabel('Sayım', fontsize=12)
ax2.set_title("Gözlemlenen vs Beklenen Sayılar", fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

### 3.4 2009 İran Seçimi Örneği

In [ ]:
# ── 2009 İran Seçimi ──────────────────────────────────────────────────────────
adaylar = ['Ahmedinejad', 'Musavi', 'Küçük Adaylar']
gozlemlenen_iran = np.array([338, 136, 30])   # anketteki oy
bildirilen_oran  = np.array([0.6329, 0.3410, 0.0261])  # seçim sonuçları
n_iran = gozlemlenen_iran.sum()

beklenen_iran = n_iran * bildirilen_oran
katkı_iran    = (gozlemlenen_iran - beklenen_iran)**2 / beklenen_iran
chi2_iran     = katkı_iran.sum()
sd_iran       = len(adaylar) - 1
p_iran        = chi2.sf(chi2_iran, df=sd_iran)

df_iran = pd.DataFrame({
    'Aday': adaylar,
    'Gözlemlenen (O)': gozlemlenen_iran,
    'Bildirilen %': (bildirilen_oran * 100).round(2),
    'Beklenen (E)': beklenen_iran.round(1),
    '(O-E)²/E': katkı_iran.round(2)
})

print("2009 İRAN SEÇİMİ — Uyum İyiliği Testi")
print(df_iran.to_string(index=False))
print("-" * 65)
print(f"χ² = {chi2_iran:.2f}   sd = {sd_iran}   p-değeri = {p_iran:.6f}")
print(f"\nKarar (α=0.05): H0 {'REDDEDİLİR' if p_iran < 0.05 else 'reddedilemez'}")
print("Yorum: Anket verileri bildirilen seçim sonuçlarıyla")
print("       uyuşmamaktadır — seçim yolsuzluğuna işaret ediyor.")

---
## 4. Bağımsızlık Ki-Kare Testi

### 4.1 Teorik Temel

İki kategorik değişken arasındaki bağımsızlığı test eder.

**Hipotezler:**
- $H_0$: İki değişken bağımsızdır
- $H_A$: İki değişken bağımlıdır

**Beklenen sayı:** $E = \dfrac{\text{satır toplamı} \times \text{sütun toplamı}}{\text{tablo toplamı}}$

**Serbestlik derecesi:** $sd = (\text{satır sayısı} - 1) \times (\text{sütun sayısı} - 1)$

### 4.2 Popüler Çocuklar Örneği

In [ ]:
# ── Veriler ──────────────────────────────────────────────────────────────────
gozlemlenen_mat = np.array([
    [63, 31, 25],   # 4. sınıf
    [88, 55, 33],   # 5. sınıf
    [96, 55, 32]    # 6. sınıf
])

satirlar = ['4. sınıf', '5. sınıf', '6. sınıf']
sutunlar  = ['Notlar', 'Popülerlik', 'Spor']

df_pop = pd.DataFrame(gozlemlenen_mat, index=satirlar, columns=sutunlar)
df_pop['Satır Toplamı'] = df_pop.sum(axis=1)

print("GÖZLEMLENEN SAYILAR:")
print(df_pop)
print(f"\nSütun Toplamı: {gozlemlenen_mat.sum(axis=0)}")
print(f"Genel Toplam : {gozlemlenen_mat.sum()}")

In [ ]:
# ── Beklenen Sayıları Hesapla ─────────────────────────────────────────────────
toplam = gozlemlenen_mat.sum()
satir_top = gozlemlenen_mat.sum(axis=1)
sutun_top = gozlemlenen_mat.sum(axis=0)

# E = satır_toplam × sütun_toplam / genel_toplam
beklenen_mat = np.outer(satir_top, sutun_top) / toplam

df_beklenen = pd.DataFrame(beklenen_mat.round(1), index=satirlar, columns=sutunlar)
print("BEKLENEN SAYILAR (E = satır_top × sütun_top / genel_top):")
print(df_beklenen)

# Koşul kontrolü: tüm hücreler ≥ 5 mi?
print(f"\nKOŞUL: Tüm beklenen sayılar ≥ 5 mi? {'✓ EVET' if (beklenen_mat >= 5).all() else '✗ HAYIR'}")

In [ ]:
# ── Ki-Kare İstatistiği ───────────────────────────────────────────────────────
chi2_pop, p_pop, sd_pop, beklenen_scipy = stats.chi2_contingency(
    gozlemlenen_mat, correction=False
)

# Manuel hesaplama
katkılar_mat = (gozlemlenen_mat - beklenen_mat)**2 / beklenen_mat
chi2_manuel = katkılar_mat.sum()

print("KI-KARE TEST İSTATİSTİĞİ")
print("\n(O-E)²/E hücre katkıları:")
df_katki = pd.DataFrame(katkılar_mat.round(4), index=satirlar, columns=sutunlar)
print(df_katki)
print("-" * 40)
print(f"χ² = {chi2_manuel:.4f}")
print(f"sd = ({len(satirlar)}-1) × ({len(sutunlar)}-1) = {sd_pop}")
print(f"p-değeri = {p_pop:.4f}")
print(f"\nKarar (α=0.05): H0 {'REDDEDİLİR' if p_pop < 0.05 else 'reddedilemez — Sınıf ve hedefler BAĞIMSIZDIR ✓'}")

In [ ]:
# ── Görselleştirme ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Yığılmış çubuk grafik
ax = axes[0]
x_pos = np.arange(len(satirlar))
genislik = 0.5
renkler_hdf = ['#4e79a7', '#f28e2b', '#59a14f']

altta = np.zeros(len(satirlar))
for i, (sutun, renk) in enumerate(zip(sutunlar, renkler_hdf)):
    degerler = gozlemlenen_mat[:, i]
    ax.bar(x_pos, degerler, genislik, bottom=altta,
           color=renk, label=sutun, alpha=0.9)
    altta += degerler

ax.set_xticks(x_pos)
ax.set_xticklabels(satirlar)
ax.set_ylabel('Öğrenci Sayısı', fontsize=12)
ax.set_title('Sınıfa Göre Hedef Dağılımı', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Sağ: Normalize edilmiş (yüzdelik) çubuk grafik
ax2 = axes[1]
satir_toplamlar = gozlemlenen_mat.sum(axis=1, keepdims=True)
yuzde = gozlemlenen_mat / satir_toplamlar * 100

altta = np.zeros(len(satirlar))
for i, (sutun, renk) in enumerate(zip(sutunlar, renkler_hdf)):
    degerler = yuzde[:, i]
    bars = ax2.bar(x_pos, degerler, genislik, bottom=altta,
                   color=renk, label=sutun, alpha=0.9)
    # Yüzde etiketleri
    for bar, val, alt in zip(bars, degerler, altta):
        if val > 4:
            ax2.text(bar.get_x() + bar.get_width()/2,
                     alt + val/2, f'%{val:.0f}',
                     ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    altta += degerler

ax2.set_xticks(x_pos)
ax2.set_xticklabels(satirlar)
ax2.set_ylabel('Yüzde (%)', fontsize=12)
ax2.set_title('Sınıfa Göre Normalize Hedef Dağılımı', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.set_ylim(0, 110)

plt.tight_layout()
plt.show()

print("Gözlem: Yüzdelik dağılımlar sınıflar arasında çok benzer → bağımsızlık")

### 4.3 Yardımcı Fonksiyon: Genel Ki-Kare Testi

In [ ]:
# ── Genel Amaçlı Ki-Kare Test Fonksiyonu ─────────────────────────────────────
def ki_kare_bagimsizlik_testi(goz_mat, satir_isimleri, sutun_isimleri, alpha=0.05):
    """
    İki kategorik değişken arasında bağımsızlık testi.
    Gözlemlenen matris, satır ve sütun isimlerini alır.
    """
    goz = np.array(goz_mat)
    toplam = goz.sum()
    satir_top = goz.sum(axis=1)
    sutun_top = goz.sum(axis=0)
    bek = np.outer(satir_top, sutun_top) / toplam

    chi2_stat = ((goz - bek)**2 / bek).sum()
    sd = (goz.shape[0] - 1) * (goz.shape[1] - 1)
    p_val = chi2.sf(chi2_stat, df=sd)

    # Sonuç
    print(f"χ² = {chi2_stat:.4f},  sd = {sd},  p-değeri = {p_val:.4f}")
    if p_val < alpha:
        print(f"→ H0 REDDEDİLİR (p={p_val:.4f} < α={alpha}): Değişkenler BAĞIMLIDIR.")
    else:
        print(f"→ H0 reddedilemez (p={p_val:.4f} ≥ α={alpha}): Değişkenler BAĞIMSIZDIR.")
    return chi2_stat, sd, p_val, bek


# ── Ek Örnek: Cinsiyet × Tercih ilişkisi ─────────────────────────────────────
print("EK ÖRNEK: Bir ankette insanların tercih ettikleri evcil hayvan ile")
print("cinsiyetleri arasında ilişki var mı?\n")

# Gözlemlenen: kedi, köpek, balık
#              erkek, kadın
ornek_veri = [
    [22, 48, 10],  # Erkek
    [38, 32, 20],  # Kadın
]

print("Gözlemlenen:")
df_ex = pd.DataFrame(ornek_veri, index=['Erkek','Kadın'], columns=['Kedi','Köpek','Balık'])
df_ex['Toplam'] = df_ex.sum(axis=1)
print(df_ex, "\n")

ki_kare_bagimsizlik_testi(
    ornek_veri,
    satir_isimleri=['Erkek', 'Kadın'],
    sutun_isimleri=['Kedi', 'Köpek', 'Balık']
)

---
## 5. Özet: Tüm Yöntemlerin Karşılaştırması

In [ ]:
# ── Kapsamlı Özet Tablosu ─────────────────────────────────────────────────────
ozet = pd.DataFrame({
    'Yöntem': [
        'Tek Oran GA',
        'Tek Oran HT',
        'İki Oran Farkı GA',
        'İki Oran Farkı HT',
        'Uyum İyiliği χ²',
        'Bağımsızlık χ²'
    ],
    'Parametre': ['p', 'p', 'p₁-p₂', 'p₁-p₂', 'Dağılım', 'Bağımsızlık'],
    'Standart Hata': [
        'sqrt(p̂(1-p̂)/n)',
        'sqrt(p₀(1-p₀)/n)',
        'sqrt(p̂₁(1-p̂₁)/n₁ + p̂₂(1-p̂₂)/n₂)',
        'sqrt(p̂h(1-p̂h)(1/n₁+1/n₂))',
        'Σ(O-E)²/E',
        'Σ(O-E)²/E'
    ],
    'Dağılım': ['Z~N(0,1)', 'Z~N(0,1)', 'Z~N(0,1)', 'Z~N(0,1)', 'χ²(k-1)', 'χ²((r-1)(c-1))'],
    'Koşul': [
        '≥10 obs başarı/başarısızlık',
        '≥10 exp başarı/başarısızlık',
        'Her grup için ≥10',
        'Her grup için ≥10',
        'Her hücre E≥5, sd>1',
        'Her hücre E≥5, sd>1'
    ]
})

print("KATEGORIK VERİLER İÇİN ÇIKARIM — ÖZET")
print("=" * 90)
print(ozet.to_string(index=False))

In [ ]:
# ── Hangi Testi Kullanmalıyım? — Karar Akış Şeması ───────────────────────────
print("""
HANGI TESTİ KULLANMALIYIM?
══════════════════════════

         Kategorik Veri
               │
    ┌──────────┴──────────┐
    │                     │
  Tek grup           İki grup
    │                     │
    │            ┌────────┴────────┐
    │          GA için         HT için
    │        İki Oran          İki Oran
    │        Farkı GA          Farkı HT
    │        (p̂₁,p̂₂         (p̂_havuz
    │         kullan)          kullan)
    │
  ┌─┴──────────────┐
  │                │
GA/HT          Çok kategori
(Z testi)           │
               ┌────┴────┐
           Tek değ.   İki değ.
          Uyum İyil.  Bağımsızlık
           Ki-Kare     Ki-Kare
           (sd=k-1)  (sd=(r-1)(c-1))
""")

---
## 6. Alıştırmalar

**Alıştırma 1:** Bir Gallup anketinde 1001 Amerikalının %11'i Halloween'e dini gerekçelerle itiraz ettiğini belirtti. %95 güven aralığı ±%3 ise, "Tüm Amerikalıların %10'dan fazlası itiraz ediyor" iddiası destekleniyor mu?

**Alıştırma 2:** İki farklı web sitesinde A/B testi yapılıyor. Site A'da 200 ziyaretçiden 32'si alışveriş yaptı, Site B'de ise 180 ziyaretçiden 45'i alışveriş yaptı. İki site arasında dönüşüm oranı açısından anlamlı fark var mı?

**Alıştırma 3:** Bir zarı 120 kez attınız ve sonuçlar şöyle: {1:18, 2:22, 3:17, 4:19, 5:25, 6:19}. Bu zar adil mi? (%5 anlamlılık düzeyi)

In [ ]:
# ── Alıştırma 1: Gallup — Güven Aralığı Yorumu ───────────────────────────────
n_gallup = 1001
p_hat_gallup = 0.11
z_star_95 = norm.ppf(0.975)
SH_g = np.sqrt(p_hat_gallup * (1 - p_hat_gallup) / n_gallup)
HP_g = z_star_95 * SH_g
GA_alt_g = p_hat_gallup - HP_g
GA_ust_g = p_hat_gallup + HP_g

print("ALIŞTıRMA 1 ÇÖZÜMÜ")
print(f"p̂ = {p_hat_gallup}, n = {n_gallup}")
print(f"HP  = {z_star_95:.2f} × {SH_g:.4f} = ±{HP_g:.4f} ≈ ±{HP_g*100:.1f}%")
print(f"%95 GA = ({GA_alt_g*100:.1f}%, {GA_ust_g*100:.1f}%)")
print(f"\nİddia: %10'dan fazlası itiraz ediyor")
print(f"GA alt sınırı %10'dan {'büyük → İDDİA DESTEKLENİYOR' if GA_alt_g > 0.10 else 'küçük → İDDİA DESTEKLENMİYOR ✗'}")
print(f"\nNeden? GA = (%{GA_alt_g*100:.1f}, %{GA_ust_g*100:.1f}) — %10'u kapsıyor!")

In [ ]:
# ── Alıştırma 2: A/B Testi ────────────────────────────────────────────────────
n_A, basari_A = 200, 32
n_B, basari_B = 180, 45
p_A = basari_A / n_A
p_B = basari_B / n_B
p_hav = (basari_A + basari_B) / (n_A + n_B)

SH_ab = np.sqrt(p_hav*(1-p_hav)*(1/n_A + 1/n_B))
Z_ab = (p_A - p_B) / SH_ab
p_ab = 2 * (1 - norm.cdf(abs(Z_ab)))

print("ALIŞTıRMA 2 ÇÖZÜMÜ — A/B Testi")
print(f"p̂_A = {basari_A}/{n_A} = {p_A:.3f}")
print(f"p̂_B = {basari_B}/{n_B} = {p_B:.3f}")
print(f"p̂_havuz = {p_hav:.3f}")
print(f"Z = {Z_ab:.3f},  p-değeri = {p_ab:.4f}")
print(f"Karar: {'Anlamlı fark VAR' if p_ab < 0.05 else 'Anlamlı fark YOK'} (α=0.05)")

In [ ]:
# ── Alıştırma 3: Zar Adil mi? ─────────────────────────────────────────────────
zar_sonuclar = np.array([18, 22, 17, 19, 25, 19])
n_zar = zar_sonuclar.sum()
bek_zar = np.array([n_zar/6] * 6)
chi2_zar = ((zar_sonuclar - bek_zar)**2 / bek_zar).sum()
sd_zar = len(zar_sonuclar) - 1
p_zar = chi2.sf(chi2_zar, df=sd_zar)

print("ALIŞTıRMA 3 ÇÖZÜMÜ — Adil Zar Testi")
print(f"Gözlemlenen: {zar_sonuclar}  (toplam={n_zar})")
print(f"Beklenen   : {bek_zar}")
print(f"χ² = {chi2_zar:.3f},  sd = {sd_zar},  p-değeri = {p_zar:.4f}")
print(f"Karar: Zar {'YANLITIR ✗' if p_zar < 0.05 else 'adil görünüyor ✓'} (α=0.05)")

---

## Notebook Özeti

Bu notebook'ta şunları öğrendik:

| Konu | Anahtar Formül | Dağılım |
|------|---------------|----------|
| Tek oran GA | $\hat{p} \pm z^* \sqrt{\hat{p}(1-\hat{p})/n}$ | Normal |
| Tek oran HT | $Z = (\hat{p}-p_0)/\sqrt{p_0(1-p_0)/n}$ | Normal |
| İki oran farkı GA | $(\hat{p}_1-\hat{p}_2) \pm z^* \cdot SH$ | Normal |
| İki oran farkı HT | $Z = (\hat{p}_1-\hat{p}_2)/SH_{havuz}$ | Normal |
| Uyum iyiliği χ² | $\chi^2 = \sum(O-E)^2/E$, $sd=k-1$ | Ki-Kare |
| Bağımsızlık χ² | $\chi^2 = \sum(O-E)^2/E$, $sd=(r-1)(c-1)$ | Ki-Kare |

**Önemli hatırlatmalar:**
- GA'da $\hat{p}$ kullan; HT'de $p_0$ kullan
- İki oran HT'de havuzlanmış oran $\hat{p}_{havuz}$ kullan
- Ki-kare testi her zaman **sağ kuyruk** testidir
- Koşullar sağlanmıyorsa → **rasgeleleştirme** yöntemi (Bölüm 6.4)